In [1]:
!pip -q install -U transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 105.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.1 MB/s eta 0:00:00


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: False
Device: CPU


In [3]:
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer
from datasets import load_dataset, concatenate_datasets


In [4]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)  # Binary classification

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
dataset = load_dataset("imdb", split="train")  # Load full train set
print("Loading..")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Loading..


In [6]:

# Ensure equal number of positive and negative samples
positive_samples = dataset.filter(lambda x: x["label"] == 1).shuffle(seed=42).select(range(1000))
negative_samples = dataset.filter(lambda x: x["label"] == 0).shuffle(seed=42).select(range(1000))

# Merge and shuffle dataset properly
balanced_dataset = concatenate_datasets([positive_samples, negative_samples]).shuffle(seed=42)

def preprocess_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=256,
        return_tensors="pt"  # Ensure tensors are properly formatted
    )

tokenized_dataset = balanced_dataset.map(preprocess_function, batched=True, remove_columns=["text"])

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
print("Sample data after tokenization:")
print(tokenized_dataset[:5])


Sample data after tokenization:
{'label': [0, 1, 0, 1, 1], 'input_ids': [[101, 1045, 2428, 18966, 2023, 3185, 1012, 1012, 1012, 1012, 3701, 2138, 1997, 1996, 2364, 3494, 999, 2027, 2024, 2119, 26838, 1010, 14337, 1010, 1998, 2969, 1011, 8857, 2111, 1012, 2027, 3480, 7955, 2105, 2068, 2652, 2037, 10021, 2208, 1012, 1996, 5107, 3896, 2020, 2204, 2021, 2054, 2204, 2024, 2027, 2065, 2045, 2024, 2053, 3494, 2008, 2017, 7532, 2007, 2030, 1037, 2466, 2240, 2008, 2003, 5875, 1012, 2572, 1045, 4011, 2000, 2022, 3407, 2043, 2122, 2048, 18224, 2111, 2633, 9530, 17421, 8585, 2037, 2293, 2005, 2169, 2060, 1029, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 2044, 3666, 2023, 3185, 1045, 2001, 3241, 1000, 2023, 2003, 4011, 2022, 1996, 1001, 1015, 15132, 2013, 2605, 1029, 1000, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1012, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 1008, 27594, 2121, 1008, 1026, 7987, 1013, 1028, 1026, 7987, 1013, 1028, 2004, 2005, 1996, 2203, 1024, 2204, 9436, 25514, 999, 20

In [8]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",  # Evaluate after each epoch
    save_strategy="epoch",
    learning_rate=5e-5,  # Increase learning rate slightly
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,  # Increase training epochs
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    lr_scheduler_type="linear",  # Helps adjust learning rate dynamically
)

print("Done")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Done


In [9]:
# Move Model to GPU Before Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Move model to GPU for faster training

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset.select(range(1500)),  # Train on 1,500 samples
    eval_dataset=tokenized_dataset.select(range(1500, 2000)),  # Eval on 500 samples
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.450905,0.345071
2,0.383869,0.447808
3,0.071645,0.514496


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=564, training_loss=0.27846074258868997, metrics={'train_runtime': 7225.6662, 'train_samples_per_second': 0.623, 'train_steps_per_second': 0.078, 'total_flos': 298051646976000.0, 'train_loss': 0.27846074258868997, 'epoch': 3.0})

In [10]:
model.save_pretrained("./fine_tuned_model")
tokenizer.save_pretrained("./fine_tuned_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./fine_tuned_model/tokenizer_config.json',
 './fine_tuned_model/tokenizer.json')

In [11]:

def classify_text(text):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Detect GPU or fallback to CPU
    model.to(device)  # Move model to the detected device

    inputs = tokenizer(text, return_tensors="pt", max_length=256, truncation=True)
    inputs = {key: value.to(device) for key, value in inputs.items()}  # Move input tensors to the same device

    outputs = model(**inputs)
    prediction = torch.argmax(outputs.logits).item()
    return "Positive" if prediction == 1 else "Negative"

In [12]:
sample_text = "This movie was amazing! I loved every moment of it."
print("Sentiment:", classify_text(sample_text))

Sentiment: Positive


In [13]:
print("Sentiment:", classify_text("I absolutely loved this film! It was fantastic."))
print("Sentiment:", classify_text("An incredible performance by the cast. A must-watch!"))
print("Sentiment:", classify_text("The storyline was beautiful and touching."))
print("Sentiment:", classify_text("This is one of the best movies I’ve ever seen."))
print("Sentiment:", classify_text("A masterpiece! Brilliant acting and a compelling plot."))

# Test Negative Sentences
print("Sentiment:", classify_text("This was the worst movie I have ever watched."))
print("Sentiment:", classify_text("Terrible acting, bad plot, and a complete waste of time."))
print("Sentiment:", classify_text("I was bored to death, nothing exciting ever happened."))
print("Sentiment:", classify_text("I regret watching this movie. It was awful."))
print("Sentiment:", classify_text("Disappointing in every way. Don't waste your time."))


Sentiment: Positive
Sentiment: Positive
Sentiment: Positive
Sentiment: Positive
Sentiment: Positive
Sentiment: Negative
Sentiment: Negative
Sentiment: Negative
Sentiment: Negative
Sentiment: Negative
